# Batch runs

A batch job halves the price and completes within 24 hours instead of streaming.
This notebook drives one end to end: write the requests, submit them, wait, read
the replies back.

Nothing here is specific to the notebook. It calls the same functions
`scripts/run.py` calls, and writes the same records live generation writes, so a
reply collected this way is indistinguishable downstream from one collected any
other way.

Only OpenAI is driven from here. Anthropic and Google have their own batch
endpoints; for those, export the file and use their console.

In [ ]:
# Import the libraries
import json
import sys
import time
from pathlib import Path

import pandas as pd

In [ ]:
# Set the working directory to the project root
if Path.cwd().name == 'notebooks':
    %cd ..

sys.path.insert(0, str(Path('scripts').resolve()))

In [ ]:
# Import the pipeline
import backends
import run
import settings
import utils

utils.make_directories()
pd.set_option('display.max_colwidth', 70)

## The model

Set the model here. It has to be one of the api models in
`config/settings.yml`, since the batch body and the price both come from its
entry in the panel.

In [ ]:
MODEL = 'gemini-3-flash'
ENDPOINT = '/v1/responses'

# what the panel holds, and how much of each is already collected
prompts = utils.read_table(settings.PROMPTS_PATH)
wanted = len(prompts) * settings.GENERATION['replicates']
panel = []
for name, s in settings.MODELS.items():
    if s['access'] != 'api':
        continue
    have = len(utils.read_lines(utils.result_path(s['id'], settings.ADAPTATION_DIR)))
    panel.append({'model': s['id'], 'provider': s['provider'],
                 'reasoning': s.get('reasoning') or 'default',
                 '$/M in': s['price']['input'], '$/M out': s['price']['output'],
                 'collected': f'{have:,} of {wanted:,}'})
display(pd.DataFrame(panel).set_index('model'))

spec = next(s for s in settings.MODELS.values() if s['id'] == MODEL)
print(f"running {MODEL}, key found in .env: "
      f"{bool(utils.api_key(spec['provider']))}")

## Write the requests

Anything already collected for this model is skipped, so this composes with a
run that stopped part way or with a live pass you started and abandoned.

In [ ]:
path, count = run.write_batch(MODEL, endpoint=ENDPOINT)

if path is None:
    print('nothing outstanding for this model')
else:
    print(f'{count:,} requests written to {path}')
    print()
    print(json.dumps(json.loads(path.read_text().splitlines()[0]), indent=2))

## What it should cost

The output figure is the guess. Run twenty live first if you have not, and put
the real average here, because output is almost the whole bill.

In [ ]:
OUTPUT_TOKENS = 211          # measured on the first full batch
INPUT_TOKENS = 24            # measured on the first full batch

price = spec['price']
standard = (count * INPUT_TOKENS * price['input']
            + count * OUTPUT_TOKENS * price['output']) / 1e6
print(f'{count:,} calls at {OUTPUT_TOKENS} output tokens each')
print(f'  standard  ${standard:,.2f}')
print(f'  batched   ${standard / 2:,.2f}')

## Submit

Uploads the file and creates the job. The id is written beside the requests, so
you can come back to this notebook tomorrow and pick the job up without having
kept the kernel alive.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=utils.api_key('openai'))

uploaded = client.files.create(file=open(path, 'rb'), purpose='batch')
job = client.batches.create(input_file_id=uploaded.id, endpoint=ENDPOINT,
                            completion_window='24h')

# name the requests after the job, so they pair with the results file the
# provider returns and a set of replies can be traced to what produced it
requests_file = run.name_after_job(MODEL, job.id)
Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt').write_text(job.id)

print(f'{job.id}  {job.status}')
print(f'requests kept at {requests_file}')

## Or pick up a job started in the console

A batch created from the provider's web console is not written to disk here, so
the status cell below has nothing to read. This lists the recent jobs on the
account and adopts one, which writes its id where the rest of the notebook
expects it. Run this instead of the submit cell above.

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=utils.api_key('openai'))

recent = client.batches.list(limit=10)
jobs = [{'id': b.id, 'status': b.status, 'endpoint': b.endpoint,
         'total': b.request_counts.total, 'completed': b.request_counts.completed,
         'failed': b.request_counts.failed,
         'created': pd.to_datetime(b.created_at, unit='s')}
        for b in recent.data]
display(pd.DataFrame(jobs))

In [ ]:
# Adopt one: paste its id here, or take the most recent
ADOPT = jobs[0]['id'] if 'jobs' in dir() and jobs else ''

if ADOPT:
    Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt').write_text(ADOPT)
    print(f'{ADOPT} adopted for {MODEL}')
    print('the status cell below will now find it')

## Wait

Re-run this cell rather than blocking the kernel. A job can take hours, and the
id is on disk, so nothing is lost by closing the notebook and coming back.

In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')

if not job_file.exists():
    print('no job submitted for this model yet, run the cell above')
else:
    job = client.batches.retrieve(job_file.read_text().strip())
    done, failed = job.request_counts.completed, job.request_counts.failed
    total = job.request_counts.total or 1
    print(f'{job.id}')
    print(f'{job.status}   {done:,} of {total:,} done, {failed} failed '
          f'({done / total:.0%})')

## Read the replies back

Writes into `results/adaptation/`, in the same shape as every other collected
reply, and prices what actually came back rather than what was estimated.

In [ ]:
job_file = Path(f'data/batches/{utils.model_slug(MODEL)}_job.txt')
job = client.batches.retrieve(job_file.read_text().strip()) if job_file.exists() else None

if job is None or job.status != 'completed':
    print(f'nothing to read yet: {job.status if job else "no job adopted"}')
else:
    results = run.batch_path(MODEL, 'output', job.id)
    results.write_bytes(client.files.content(job.output_file_id).read())
    print(f'{results} downloaded')

    first = json.loads(results.read_text().splitlines()[0])
    print(f"first custom_id: {first['custom_id']}")

    backends.USAGE.update(calls=0, input=0, output=0)
    read, failed = run.read_batch(MODEL, results)

    usage, cost = backends.USAGE, backends.spent(MODEL)
    print(f'\n{read:,} replies read, {failed} failed')
    print(f'{usage["input"]:,} input and {usage["output"]:,} output tokens')
    print(f'${cost:,.2f} at the standard rate, ${cost / 2:,.2f} batched')
    print(f'{usage["output"] / max(read - failed, 1):.0f} output tokens a reply, '
          f'against the {OUTPUT_TOKENS} assumed above')

## Check what arrived

Empty replies are the failure to watch for on a reasoning model: reasoning
tokens count against the output cap, so a reply can come back blank having
spent its whole budget thinking.

In [ ]:
collected = utils.read_lines(utils.result_path(MODEL, settings.ADAPTATION_DIR))
prompts = utils.read_table(settings.PROMPTS_PATH)

if collected.empty:
    print(f'nothing collected for {MODEL} yet')
else:
    blank = int((collected['response'].astype(str).str.strip() == '').sum())
    errored = int((collected['error'].astype(str).str.strip() != '').sum())
    print(f'{len(collected):,} replies, {blank} empty, {errored} errored')
    print(f"{collected['prompt_id'].nunique():,} of {len(prompts):,} prompts covered")

    display(collected.merge(prompts[['prompt_id', 'condition', 'prompt']],
                            on='prompt_id')[['condition', 'prompt',
                                             'response']].head(5))

## Then

Judge them:

```
python scripts/evaluate.py --backend vllm --model {MODEL} --batch-size 64
```